# 10.7 · Responsible AI: fairness, privacy, explainability, governance

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/07-responsible-ai/lesson.ipynb)

Read the [README](README.md) first. This notebook measures whether the churn model treats UK and non-UK customers alike, shows how much a small group's numbers can be trusted, removes personal data from text before it reaches an LLM, and writes a model card.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


## 1. Fairness metrics, by hand

The README's ten customers. Group A: six customers, group B: four. `flag` = the model would send a voucher; `churned` = what happened.

In [2]:
toy = pd.DataFrame({"group":   list("AAAAAABBBB"),
                    "flag":    [1, 1, 1, 0, 0, 0, 1, 0, 0, 0],
                    "churned": [1, 1, 0, 1, 0, 0, 1, 1, 0, 0]})
for name, g in toy.groupby("group"):
    churn = g["churned"] == 1
    print(f"group {name}: flag rate {g['flag'].mean():.2f}, "
          f"recall {(g['flag'][churn] == 1).mean():.2f}, "
          f"false positive rate {(g['flag'][~churn] == 1).mean():.2f}")

group A: flag rate 0.50, recall 0.67, false positive rate 0.33
group B: flag rate 0.25, recall 0.50, false positive rate 0.00


## 2. The real groups

ShopWave has no data on age, gender or ethnicity, and should not. The only customer attribute is country. We use **UK vs non-UK** as an example of a group the business might want treated equally (non-UK customers are often wholesalers). The method is the same for any group.

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.base import clone
cust = cm.build_customer_table(cm.load_sales())
train, test = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
test = test.assign(p=bundle["pipeline"].predict_proba(test[cm.FEATURES])[:, 1],
                   group=np.where(test["country"] == "United Kingdom", "UK", "non-UK"))
print(test["group"].value_counts().to_string())

group
UK        749
non-UK     81


At the saved threshold (0.07, for a £5 voucher), and at the threshold that would be cheapest for a **£20 voucher**, chosen on cross-validated training predictions as in 05.10:

In [4]:
p_cv = cross_val_predict(clone(bundle["pipeline"]), train[cm.FEATURES], train["churned"],
                         cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]
grid = np.round(np.arange(0.01, 1, 0.01), 2)
t20 = float(grid[np.argmin([cm.expected_cost(train["churned"], p_cv, t, c_voucher=20) for t in grid])])
print("cheapest threshold for a £20 voucher:", t20)

cheapest threshold for a £20 voucher: 0.35


In [5]:
def group_metrics(df, threshold):
    """Flag rate, recall (true positive rate) and false positive rate per group."""
    rows = {}
    for name, g in df.groupby("group"):
        flag, churn = g["p"] >= threshold, g["churned"] == 1
        rows[name] = {"customers": len(g), "churners": int(churn.sum()), "flag rate": flag.mean(),
                      "recall": (flag & churn).sum() / churn.sum(),
                      "false positive rate": (flag & ~churn).sum() / (~churn).sum(),
                      "avg predicted": g["p"].mean(), "actual churn": churn.mean()}
    return pd.DataFrame(rows).T

for t in [bundle["threshold"], t20]:
    print(f"\nthreshold {t}")
    print(group_metrics(test, t).round(3).to_string())


threshold 0.07
        customers  churners  flag rate  recall  false positive rate  avg predicted  actual churn
UK          749.0     332.0      0.935   0.997                0.885          0.432         0.443
non-UK       81.0      30.0      0.951   1.000                0.922          0.426         0.370

threshold 0.35
        customers  churners  flag rate  recall  false positive rate  avg predicted  actual churn
UK          749.0     332.0      0.658   0.846                0.508          0.432         0.443
non-UK       81.0      30.0      0.617   0.800                0.510          0.426         0.370


At £5 almost everyone gets a voucher, so there is nothing to be unfair about. At £20 the gaps are small: flag rates differ by 4 points (demographic parity), recall by 5 (equal opportunity), the false positive rate hardly at all. Calibration differs a little more: for UK customers the average prediction matches the actual churn (0.43 vs 0.44), for non-UK customers it is too high (0.43 vs 0.37), so non-UK customers are slightly more likely to get a voucher they did not need.

## 3. Is the gap real?

The non-UK group has 81 test customers and only 30 churners. Resample them with replacement 1,000 times (a **bootstrap**) and see how much its recall moves by chance alone.

In [6]:
rng = np.random.default_rng(0)
non_uk = test[test["group"] == "non-UK"]
recalls = []
for _ in range(1000):
    s = non_uk.sample(len(non_uk), replace=True, random_state=int(rng.integers(1e9)))
    churn = s["churned"] == 1
    recalls.append(((s["p"] >= t20) & churn).sum() / churn.sum())
low, high = np.percentile(recalls, [2.5, 97.5])
uk_recall = group_metrics(test, t20).loc["UK", "recall"]
print(f"non-UK recall: 95% interval {low:.2f} to {high:.2f};  UK recall {uk_recall:.2f}")

non-UK recall: 95% interval 0.65 to 0.93;  UK recall 0.85


The UK value sits comfortably inside the non-UK interval. With 30 churners we cannot tell the groups apart, in either direction. The honest conclusion is "no evidence of a gap, and too little data to rule out a moderate one", and the right action is to keep measuring as more non-UK customers arrive.

## 4. Privacy: what personal data is there?

In [7]:
raw_columns = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
print("raw transaction columns:", raw_columns)
counts = cust["country"].value_counts()
print(f"countries with fewer than 5 customers: {(counts < 5).sum()} "
      f"({counts[counts < 5].sum()} customers)")

raw transaction columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


countries with fewer than 5 customers: 17 (30 customers)


No names, emails or addresses: the Customer ID is a **pseudonym**. But pseudonymous is not anonymous. A customer who is one of only a handful from their country, with their exact order dates, may be identifiable to someone who knows them. Under GDPR, pseudonymised data is still personal data.

Support emails (Module 09) are worse: they contain names, emails, phone numbers and addresses. Remove what the task does not need **before** text is sent to an LLM, especially a hosted one:

In [8]:
import re
PATTERNS = {"EMAIL": r"[\w.+-]+@[\w-]+\.[\w.]+", "PHONE": r"\+?\d[\d\s-]{8,}\d",
            "POSTCODE": r"\b[A-Z]{1,2}\d[A-Z\d]? ?\d[A-Z]{2}\b", "ORDER": r"\b\d{6}\b"}

def redact(text):
    for label, pattern in PATTERNS.items():
        text = re.sub(pattern, f"[{label}]", text)
    return text

email = ("Hi, I'm Priya Shah (priya.shah@example.com, 07700 900123). Order 536365 arrived broken "
         "at 12 Hill Road, Leeds LS6 2AB. Please refund.")
print(redact(email))

Hi, I'm Priya Shah ([EMAIL], [PHONE]). Order [ORDER] arrived broken at 12 Hill Road, Leeds [POSTCODE]. Please refund.


The name and the street address got through: they do not follow a simple pattern. Real systems add a named-entity model (09.5) and, above all, ask whether the LLM needs the text at all. The order number was replaced too, which a support bot might actually need: decide per task what to keep.

## 5. A model card

In [9]:
fair = group_metrics(test, t20)
card = f"""# Model card: ShopWave churn model
**Purpose.** Rank existing customers by risk of not ordering in the next 91 days, to target win-back vouchers.
**Not for.** Credit, pricing, refusing service, or any decision about a person beyond a marketing offer.
**Data.** {bundle["data"]}. Features: {", ".join(bundle["features"])}. No demographic data.
**Model.** Scaled logistic regression, trained {bundle["trained"]} (scikit-learn {bundle["sklearn_version"]}).
**Performance.** Test AUC {bundle["metrics"]["test_auc"]}. Threshold {bundle["threshold"]} for a £5 voucher; {t20} for £20.
**Fairness (at {t20}).** Recall UK {fair.loc["UK", "recall"]:.2f} vs non-UK {fair.loc["non-UK", "recall"]:.2f}
(non-UK: {int(fair.loc["non-UK", "churners"])} churners, difference within chance).
**Limits.** Calibration drifts with the season (10.6); retrain and re-check thresholds before peak season.
**Oversight.** Retention team reviews each campaign list; customers can opt out of marketing.
**Owner.** Data team. Review every quarter."""
print(card)

# Model card: ShopWave churn model
**Purpose.** Rank existing customers by risk of not ordering in the next 91 days, to target win-back vouchers.
**Not for.** Credit, pricing, refusing service, or any decision about a person beyond a marketing offer.
**Data.** 3,317 customers, orders before 2011-09-01, churn = no order in next 91 days. Features: orders, spend, products, tenure_days, days_since_last. No demographic data.
**Model.** Scaled logistic regression, trained 2026-10-28 (scikit-learn 1.9.1).
**Performance.** Test AUC 0.741. Threshold 0.07 for a £5 voucher; 0.35 for £20.
**Fairness (at 0.35).** Recall UK 0.85 vs non-UK 0.80
(non-UK: 30 churners, difference within chance).
**Limits.** Calibration drifts with the season (10.6); retrain and re-check thresholds before peak season.
**Oversight.** Retention team reviews each campaign list; customers can opt out of marketing.
**Owner.** Data team. Review every quarter.


## Quiz answers

1. Demographic parity compares how often each group is flagged; equal opportunity compares recall, how often each group's actual churners are caught.
2. Too few churners (30): random variation alone moves its recall by about ±0.15.
3. No: pseudonymous data can often be linked back to a person, so it is still personal data under GDPR.
4. Remove or mask personal details the task does not need, and prefer not sending the text at all if possible.
5. Minimal risk: a marketing offer. It would become high-risk if the same score decided credit, insurance pricing or access to essential services.

**Next:** [10.P Mini-project: deploy the churn model and the chatbot](../project/)